In [1]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain.tools import tool
from langchain.agents import create_agent

load_dotenv()

/home/asus/Desktop/agentic_ai_project/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [2]:
PRODUCTS = {
    "wireless headphones": {
        "price": 79.99,
        "description": "Over-ear Bluetooth, 30-hr battery, active noise cancellation."
    },
    "smart watch": {
        "price": 199.99,
        "description": "Tracks heart rate and sleep. 5-day battery, water-resistant."
    },
    "mechanical keyboard": {
        "price": 129.00,
        "description": "Tenkeyless, Cherry MX Brown switches, per-key RGB."
    },
    "laptop stand": {
        "price": 34.99,
        "description": "Adjustable aluminum, fits 11-17 inch laptops, folds flat."
    },
}

REVIEWS = {
    "wireless headphones": {"reviews": 1262, "rating": 4.6},
    "smart watch": {"reviews": 340, "rating": 3.9},
    "mechanical keyboard": {"reviews": 67, "rating": 4.8},
    "laptop stand": {"reviews": 781, "rating": 4.5},
}

@tool
def get_product(name: str) -> str:
    """Look up a product by name and return its price, rating, stock, and description"""
    p = PRODUCTS.get(name.lower())
    if not p:
        return f"Product not found. Available: {', '.join(PRODUCTS)}"
    return str(p)

@tool
def get_review(name: str) -> str:
    """Look up a product review by a product name. Return the product name, number of reviews and rating"""
    r = REVIEWS.get(name.lower())
    if not r:
        return "Review not available for this product."
    return str(r)

In [3]:
llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

agent = create_agent(
    llm,
    tools=[get_product, get_review],
    system_prompt="You are a helpful product assistant for an online tech store.",
)

In [4]:
def ask(question: str):
    result = agent.invoke(
    {"messages": [{"role": "user", "content": question}]}
    )
    print(result["messages"][-1].content)

In [5]:
ask("what is the price of wireless headphone")

The price of the wireless headphones is $79.99. They are over-ear Bluetooth headphones with a 30-hour battery life and active noise cancellation.


In [7]:
ask("what is the reviews on this product?")

I need to know the name of the product to look up its review. Can you please provide the product name?


In [9]:
from langgraph.checkpoint.memory import InMemorySaver

llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

agent2 = create_agent(
    llm,
    tools = [get_product, get_review],
    system_prompt = "You are a helpful product assistant for an online tech store.",
    checkpointer = InMemorySaver()
)

def ask2(question: str):
    config = {"configurable": {"thread_id": "thread-1"}}
    result = agent2.invoke(
        {"messages": [{"role": "user", "content": question}]},
        config
    )
    print(result["messages"][-1].content)

In [10]:
ask2("what is the price of wireless headphone")

The price of the wireless headphones is $79.99. They are over-ear Bluetooth headphones with a 30-hour battery life and active noise cancellation.


In [11]:
ask2("what is the reviews on this product?")

The wireless headphones have received 1262 reviews with an average rating of 4.6.
